In [ ]:
# %%
import warnings
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

BASE_PATH = '/Users/egorilin/Desktop/COMA'
BASE = Path(BASE_PATH)

RANDOM_STATE = 42
TEST_SIZE = 0.2

CB_KW = dict(iterations=500, learning_rate=0.05, depth=6,
             verbose=False, random_seed=RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

TARGET_COLUMNS = [
    'S. aureus ATCC 43300 Activity MIC, мг/л',
    'S. aureus ATCC 43300 Activity MBC, мг/л',
    'E. coli ATCC 25922 Activity MIC, мг/л',
    'E. coli ATCC 25922 Activity MBC, мг/л',
]

# 'biocides' — 12 фич (как Cell 6, 108 mol); 'conc' — 15 фич (82 mol)
DATASET_MODE = 'biocides'

FEATURE_COLS_FULL = [
    'Molecular weight', 'LogP', 'mmff94_dipole', 'gasteiger_dipole', 'eem2015bm_dipole',
    'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85',
    'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729',
    'Molformer_337', 'Molformer_686',
]
FEATURE_COLS_BIO = [
    'Molecular weight', 'LogP',
    'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85',
    'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729',
    'Molformer_337', 'Molformer_686',
]
FEATURE_COLS = FEATURE_COLS_BIO if DATASET_MODE == 'biocides' else FEATURE_COLS_FULL

DESC_OUT_DIR = BASE / 'descriptors_test_predictions'
EMB_OUT_DIR  = BASE / 'embeddings_test_predictions'
DESC_OUT_DIR.mkdir(exist_ok=True)
EMB_OUT_DIR.mkdir(exist_ok=True)

print(f'Dataset mode: {DATASET_MODE}  |  {len(FEATURE_COLS)} фич')
print(f'Output dirs:\n  {DESC_OUT_DIR}\n  {EMB_OUT_DIR}')

In [ ]:
# %%
# Таблица с таргетами
target = pd.read_excel(f'{BASE_PATH}/Data_biocides.xlsx')

# Дескрипторный датасет (для подхода 1)
desc_df = pd.read_excel(f'{BASE_PATH}/Data_molformer_conc.xlsx')

# Молекулярные фичи (768-dim) для энкодера (для подхода 2)
data_drop = pd.read_excel(f'{BASE_PATH}/Test_molformer_original_biocides.xlsx').drop(
    ['Unnamed: 0'], axis=1
)

# Таблица с диполями
dipoles_df = pd.read_csv(f'{BASE_PATH}/Train_dipole_moments.csv')

print(f'target:     {target.shape}')
print(f'desc_df:    {desc_df.shape}  | cols: {list(desc_df.columns)[:8]}...')
print(f'data_drop:  {data_drop.shape}')
print(f'dipoles_df: {dipoles_df.shape}  | cols: {list(dipoles_df.columns)}')

# --- Мердж диполей в desc_df ---
DIPOLE_COLS = ['mmff94_dipole', 'gasteiger_dipole', 'eem2015bm_dipole']

# Автоопределение общего ключа (SMILES / smiles / Name / Compound и т.п.)
candidates = ['SMILES', 'smiles', 'Smiles',
              'Name', 'name', 'Compound', 'compound', 'ID', 'id']
merge_key = None
for k in candidates:
    if k in desc_df.columns and k in dipoles_df.columns:
        merge_key = k
        break

if merge_key is None:
    common = set(desc_df.columns) & set(dipoles_df.columns)
    common -= set(DIPOLE_COLS) | set(TARGET_COLUMNS)
    print(f'⚠️  Не нашёл стандартный ключ. Общие колонки: {common}')
    raise ValueError(
        'Укажи руками: merge_key = "<имя колонки>" и убери этот raise. '
        f'Колонки desc_df: {list(desc_df.columns)[:20]}; '
        f'колонки dipoles_df: {list(dipoles_df.columns)}'
    )

print(f'\n✓ Ключ для мерджа: "{merge_key}"')

# проверим, какие диполи реально есть в файле
have_dipoles = [c for c in DIPOLE_COLS if c in dipoles_df.columns]
miss_dipoles = [c for c in DIPOLE_COLS if c not in dipoles_df.columns]
print(f'  Диполи в файле: {have_dipoles}')
if miss_dipoles:
    print(f'  ⚠️  Отсутствуют в файле диполей: {miss_dipoles}')

# убираем из desc_df старые (если вдруг частично есть) и мерджим
desc_df = desc_df.drop(columns=[c for c in DIPOLE_COLS if c in desc_df.columns])
desc_df = desc_df.merge(
    dipoles_df[[merge_key] + have_dipoles],
    on=merge_key, how='left'
)

# отчёт по покрытию
print('\nПокрытие диполей после мерджа:')
for c in have_dipoles:
    n_nan = desc_df[c].isna().sum()
    print(f'  {c}: {len(desc_df) - n_nan}/{len(desc_df)} заполнено  ({n_nan} NaN)')

# --- Проверка колонок ---
missing_desc = [c for c in FEATURE_COLS if c not in desc_df.columns]
if missing_desc:
    raise ValueError(f'Нет колонок в desc_df после мерджа: {missing_desc}')

missing_tg_desc = [c for c in TARGET_COLUMNS if c not in desc_df.columns]
if missing_tg_desc:
    raise ValueError(f'Нет таргетов в desc_df: {missing_tg_desc}')

missing_tg = [c for c in TARGET_COLUMNS if c not in target.columns]
if missing_tg:
    raise ValueError(f'Нет таргетов в Data_biocides.xlsx: {missing_tg}')

print('\n✓ Все нужные колонки на месте')
print(f'  Финальные FEATURE_COLS ({len(FEATURE_COLS)}): {FEATURE_COLS}')